In [ ]:
class ContinuumTile:

    """
    Stores one local image chunk.
    """

    def __init__(
        self,
        feature_data,
        continuum_data,
        x1,
        x2,
        y1,
        y2,
        tile_id
    ):

        self.feature = feature_data
        self.continuum = continuum_data

        self.x1 = x1
        self.x2 = x2
        self.y1 = y1
        self.y2 = y2

        self.tile_id = tile_id

        self.shift = None
        self.scale_factor = None

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import glob
from astropy.io import fits
from matplotlib.widgets import Slider

from reproject import reproject_interp
from scipy.ndimage import fourier_shift
from skimage.registration import phase_cross_correlation

from astropy.io import fits
from astropy.wcs import WCS
from photutils.centroids import centroid_quadratic
%matplotlib widget
class ContinuumSubtraction:

    """
    Tools for continuum subtraction from filters.
    """

    # =========================================================
    # INITIALIZATION
    # =========================================================

    def __init__(self):

        self.images = {}
        self.headers = {}
        self.files = {}
        self.wcs = {}
    # =========================================================
    # LOAD FITS IMAGE
    # =========================================================




    def load_image(self, name, filename):

        """
        Load JWST FITS image properly with WCS.
        """

        hdul = fits.open(filename)

        self.files[name] = filename

        # --------------------------------------------------------
        # TRY SCI EXTENSION FIRST
        # --------------------------------------------------------

        try:

            data = hdul['SCI'].data.astype(float)

            header = hdul['SCI'].header

            wcs = WCS(hdul['SCI'].header, hdul)

        # --------------------------------------------------------
        # OTHERWISE USE PRIMARY
        # --------------------------------------------------------

        except:
            print('sci failed')
            data = hdul[0].data.astype(float)

            header = hdul[0].header

            wcs = WCS(hdul[0].header)

        self.images[name] = data
        self.headers[name] = header
        self.wcs[name] = wcs

        print(f'Loaded: {name}')

    # =========================================================
    # APPLY CIRCULAR MASK
    # =========================================================

    def circular_mask(
        self,
        image_name,
        x_center,
        y_center,
        radius
    ):

        data = self.images[image_name].copy()

        yy, xx = np.indices(data.shape)

        r = np.sqrt(
            (xx - x_center)**2 +
            (yy - y_center)**2
        )

        mask = r <= radius

        data[mask] = np.nan

        self.images[image_name] = data

        print(f'Masked {image_name}')

    # =========================================================
    # REPROJECT IMAGE
    # =========================================================
    def check_alignment(
        self,
        image1,
        image2
    ):

        """
        Print basic alignment diagnostics.
        """

        data1 = self.images[image1]
        data2 = self.images[image2]

        print('----------------------------------')
        print('Alignment Diagnostics')
        print('----------------------------------')
        print(f'{image1} shape: {data1.shape}')
        print(f'{image2} shape: {data2.shape}')

        h1 = self.headers[image1]
        h2 = self.headers[image2]

        try:

            pix1 = abs(h1['CDELT1'])
            pix2 = abs(h2['CDELT1'])

            print(f'{image1} pixel scale: {pix1}')
            print(f'{image2} pixel scale: {pix2}')

        except:

            print('Could not determine CDELT1')

        print('----------------------------------')
    # =========================================================
    # MEASURE SUBPIXEL SHIFT
    # =========================================================
    def measure_shift(
        self,
        reference_image,
        moving_image,
        x_star,
        y_star,
        box_size=50,
        show_cutout=True
    ):

        """
        Measure subpixel shift using stellar centroids.
        """

        ref = self.images[reference_image]
        mov = self.images[moving_image]

        # ========================================================
        # CUTOUT
        # ========================================================

        half = box_size // 2

        x1 = int(x_star - half)
        x2 = int(x_star + half)

        y1 = int(y_star - half)
        y2 = int(y_star + half)

        ref_cut = ref[y1:y2, x1:x2]
        mov_cut = mov[y1:y2, x1:x2]

        # ========================================================
        # REMOVE NaNs
        # ========================================================

        ref_cut = np.nan_to_num(ref_cut, nan=0.0)
        mov_cut = np.nan_to_num(mov_cut, nan=0.0)

        # ========================================================
        # CENTROIDS
        # ========================================================

        ref_x, ref_y = centroid_quadratic(ref_cut)
        mov_x, mov_y = centroid_quadratic(mov_cut)

        # ========================================================
        # COMPUTE SHIFT
        # ========================================================

        shift_x = ref_x - mov_x
        shift_y = ref_y - mov_y

        shift = [shift_y, shift_x]

        # ========================================================
        # PRINT
        # ========================================================

        print('----------------------------------')
        print('Centroid Registration')
        print('----------------------------------')

        print(f'Reference centroid : ({ref_x:.4f}, {ref_y:.4f})')
        print(f'Moving centroid    : ({mov_x:.4f}, {mov_y:.4f})')

        print(f'Shift (y,x) = ({shift_y:.4f}, {shift_x:.4f})')

        print('----------------------------------')

        # ========================================================
        # DIAGNOSTIC PLOT
        # ========================================================

        if show_cutout:

            vmax = np.nanpercentile(
                np.concatenate([
                    ref_cut.ravel(),
                    mov_cut.ravel()
                ]),
                99.5
            )

            fig, axes = plt.subplots(
                1,
                2,
                figsize=(8,4)
            )

            axes[0].imshow(
                ref_cut,
                origin='lower',
                cmap='gray',
                vmax=vmax
            )

            axes[0].scatter(
                ref_x,
                ref_y,
                s=100,
                marker='+'
            )

            axes[0].set_title('Reference')

            axes[1].imshow(
                mov_cut,
                origin='lower',
                cmap='gray',
                vmax=vmax
            )

            axes[1].scatter(
                mov_x,
                mov_y,
                s=100,
                marker='+'
            )

            axes[1].set_title('Moving')

            for ax in axes:
                ax.axis('off')

            plt.tight_layout()
            plt.show()

        return shift    # =========================================================
        # APPLY FOURIER SHIFT
        # =========================================================

    def apply_shift(
    self,
    image_name,
    shift,
    output_name=None
    ):

        """
        Apply subpixel Fourier shift while preserving NaNs.
        """

        print('Applying shift...')

        if output_name is None:
            output_name = f'{image_name}_shifted'

        data = self.images[image_name]

        # ========================================================
        # VALID PIXEL MASK
        # ========================================================

        valid = np.isfinite(data).astype(float)

        # ========================================================
        # FILL NaNs
        # ========================================================

        filled = np.nan_to_num(data, nan=0.0)

        # ========================================================
        # FOURIER SHIFT IMAGE
        # ========================================================

        shifted_fft = fourier_shift(
            np.fft.fftn(filled),
            shift
        )

        shifted = np.fft.ifftn(
            shifted_fft
        ).real

        # ========================================================
        # SHIFT VALIDITY MASK
        # ========================================================

        mask_fft = fourier_shift(
            np.fft.fftn(valid),
            shift
        )

        shifted_mask = np.fft.ifftn(
            mask_fft
        ).real

        # ========================================================
        # RESTORE NaNs
        # ========================================================

        shifted[shifted_mask < 0.95] = np.nan

        # ========================================================
        # SAVE
        # ========================================================

        self.images[output_name] = shifted

        self.headers[output_name] = (
            self.headers[image_name].copy()
        )

        print(f'Applied shift to {image_name}')

        print(
            f'Finite pixels: '
            f'{np.sum(np.isfinite(shifted))}'
        )

    # =========================================================
    # CONTINUUM SUBTRACTION
    # =========================================================

    def continuum_subtract(
        self,
        f187_name,
        continuum_name,
        scale_factor,
        output_name='cont_subtracted'
    ):

        subtracted = (
            self.images[f187_name] -
            scale_factor * self.images[continuum_name]
        )

        self.images[output_name] = subtracted
        self.headers[output_name] = self.headers[
            f187_name
        ].copy()

        print(f'Created: {output_name}')

    # =========================================================
    # SAVE FITS
    # =========================================================

    def save_fits(
        self,
        image_name,
        output_file
    ):

        hdu = fits.PrimaryHDU(
            data=self.images[image_name],
            header=self.headers[image_name]
        )

        hdu.writeto(
            output_file,
            overwrite=True
        )

        print(f'Saved: {output_file}')

    # =========================================================
    # INTERACTIVE INSPECTION
    # =========================================================

    def inspect_continuum_subtraction(
        self,
        feature_name,
        continuum_name,
        initial_scale=1.072,
        zoom_size=1000,
        mask_x=None,
        mask_y=None,
        mask_radius=None,
        show_all=False
    ):

        # -----------------------------------------------------
        # COPY DATA
        # -----------------------------------------------------

        feature = self.images[feature_name].copy()
        cont = self.images[continuum_name].copy()

        # -----------------------------------------------------
        # OPTIONAL MASK
        # -----------------------------------------------------

        if (
            mask_x is not None and
            mask_y is not None and
            mask_radius is not None
        ):

            yy, xx = np.indices(feature.shape)

            r = np.sqrt(
                (xx - mask_x)**2 +
                (yy - mask_y)**2
            )

            mask = r <= mask_radius

            feature[mask] = np.nan
            cont[mask] = np.nan

        # -----------------------------------------------------
        # CENTRAL CUTOUT
        # -----------------------------------------------------
        if zoom_size is not None:
            ny, nx = feature.shape

            x_center = nx // 2
            y_center = ny // 2

            x1 = x_center - zoom_size // 2
            x2 = x_center + zoom_size // 2

            y1 = y_center - zoom_size // 2
            y2 = y_center + zoom_size // 2

            feature_cut = feature[y1:y2, x1:x2]
            cont_cut = cont[y1:y2, x1:x2]
        else:
            feature_cut = feature
            cont_cut = cont

        # -----------------------------------------------------
        # INITIAL MODEL
        # -----------------------------------------------------

        continuum = initial_scale * cont_cut

        subtracted = feature_cut - continuum

        # -----------------------------------------------------
        # NORMALIZATION
        # -----------------------------------------------------
        if show_all:
            combined = np.concatenate([
                feature_cut[np.isfinite(feature_cut)].ravel(),
                continuum[np.isfinite(continuum)].ravel(),
                cont_cut[np.isfinite(cont_cut)].ravel()
            ])

            vmin = np.percentile(combined, 1)
            vmax = np.percentile(combined, 99.7)
            fig, axes = plt.subplots(
            2,
            2,
            figsize=(8, 8)
            )

            axes = axes.ravel()

        else:
            vmax = np.percentile(feature_cut[np.isfinite(feature_cut)].ravel(), 99.7)
            vmin = np.percentile(feature_cut[np.isfinite(feature_cut)].ravel(), 1)

            fig, axes = plt.subplots(figsize=(6, 6))



        sub_v = np.nanpercentile(
            np.abs(subtracted),
            99
        )

        # -----------------------------------------------------
        # FIGURE
        # -----------------------------------------------------


        plt.subplots_adjust(bottom=0.15)

        # -----------------------------------------------------
        # F187N
        # -----------------------------------------------------
        if show_all:
            im0 = axes[0].imshow(
                feature_cut,
                origin='lower',
                cmap='gray',
                vmin=vmin,
                vmax=vmax
            )

            axes[0].set_title('feature')
            axes[0].axis('off')

            # -----------------------------------------------------
            # CONTINUUM
            # -----------------------------------------------------

            im1 = axes[1].imshow(
                continuum,
                origin='lower',
                cmap='gray',
                vmin=vmin,
                vmax=vmax
            )

            title1 = axes[1].set_title(
                f'Continuum = {initial_scale:.5f}'
            )

            axes[1].axis('off')

            # -----------------------------------------------------
            # SUBTRACTED
            # -----------------------------------------------------

            im2 = axes[2].imshow(
                subtracted,
                origin='lower',
                cmap='RdBu_r',
                vmin=-sub_v,
                vmax=sub_v
            )

            title2 = axes[2].set_title(
                'feature - Continuum'
            )

            axes[2].axis('off')

            # -----------------------------------------------------
            # F150W
            # -----------------------------------------------------

            im3 = axes[3].imshow(
                cont_cut,
                origin='lower',
                cmap='gray',
                vmin=vmin,
                vmax=vmax
            )

            axes[3].set_title('Continuum Image')
            axes[3].axis('off')

            # -----------------------------------------------------
            # COLORBARS
            # -----------------------------------------------------

            plt.colorbar(
                im0,
                ax=axes[0],
                fraction=0.046
            )

            plt.colorbar(
                im1,
                ax=axes[1],
                fraction=0.046
            )

            plt.colorbar(
                im2,
                ax=axes[2],
                fraction=0.046
            )

            plt.colorbar(
                im3,
                ax=axes[3],
                fraction=0.046
            )

        else:
            im2 = axes.imshow(
                subtracted,
                origin='lower',
                cmap='RdBu_r',
                vmin=-sub_v,
                vmax=sub_v
            )

            title2 = axes.set_title(
                'feature - Continuum'
            )

            axes.axis('off')

        # -----------------------------------------------------
        # SLIDER
        # -----------------------------------------------------

        ax_slider = plt.axes(
            [0.2, 0.05, 0.6, 0.03]
        )

        scale_slider = Slider(
            ax=ax_slider,
            label='Scale Factor',
            valmin=0.01,
            valmax=2,
            valinit=initial_scale,
            valstep=0.001
        )

        # -----------------------------------------------------
        # UPDATE
        # -----------------------------------------------------

        def update(val):

            scale = scale_slider.val

            continuum_new = scale * cont_cut

            subtracted_new = (
                feature_cut -
                continuum_new
            )
            if show_all:
                im1.set_data(continuum_new)
                title1.set_text(f'Continuum = {scale:.5f}')
            im2.set_data(subtracted_new)

            sub_v_new = np.nanpercentile(
                np.abs(subtracted_new),
                99
            )

            im2.set_clim(
                -sub_v_new,
                sub_v_new
            )


            title2.set_text(
                f'Median Residual = '
                f'{np.nanmedian(subtracted_new):.5e}'
            )

            fig.canvas.draw_idle()

        scale_slider.on_changed(update)

        plt.show()

In [ ]:
pa_cont_file = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f150w_i2d_anchor.fits'
pa_feature_file = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_i2d_anchor.fits'
ha_cont_file = '/project/galaxies/tjuchau/data_files/HST/ngc5194/F689M_HST_WFC3_UVIS_IVM_drc.fits'
ha_feature_file = '/project/galaxies/tjuchau/data_files/HST/ngc5194/F658N_HST_ACS_WFC_IVM_drc.fits'

#initialize pa-a object
pcs = ContinuumSubtraction()
#load images
pcs.load_image('cont', pa_cont_file)
pcs.load_image('feature', pa_feature_file)

#measure subpixel offset
shift = pcs.measure_shift('feature', 'cont', 5960, 5744, box_size=50, show_cutout=False)

#apply corrective shift
pcs.apply_shift('cont', shift, output_name='feature_alligned')

'''
#initialize H-a object
hcs = PaAlphaContinuumSubtractor()
#load images
hcs.load_image('cont', ha_cont_file)
hcs.load_image('feature', ha_feature_file)

#measure subpixel offset
shift = hcs.measure_shift('feature', 'cont', 8502, 5970, box_size=50, show_cutout=False)

#apply corrective shift
hcs.apply_shift('cont', shift, output_name='feature_alligned')
'''



In [ ]:
# Interactive Paschen-alpha inspection
pcs.inspect_continuum_subtraction(
    feature_name='feature',
    continuum_name='cont',
    initial_scale=1.072,
    zoom_size=200,
    mask_x=5200,
    mask_y=5200,
    mask_radius=250,
    #show_all=True
)

In [ ]:
# Interactive inspection
hcs.inspect_continuum_subtraction(
    feature_name='feature',
    continuum_name='cont',
    initial_scale=0.251,
    zoom_size=None,
    mask_x=None,
    mask_y=None,
    mask_radius=None
)

In [ ]:
pcs.continuum_subtract('feature', 'cont', 1.072, output_name='cont_subtracted')
pcs.save_fits('cont_subtracted', '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_continuum_subtracted.fits')

hcs.continuum_subtract('feature', 'cont', 0.251, output_name='cont_subtracted')
hcs.save_fits('cont_subtracted', '/project/galaxies/tjuchau/data_files/HST/ngc5194/F658N_HST_continuum_subtracted.fits')


In [ ]:
Ha = fits.open('/project/galaxies/tjuchau/data_files/HST/ngc5194/F658N_HST_continuum_subtracted.fits')[0].data
Pa = fits.open('/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_continuum_subtracted.fits')[0].data
ha_v = np.nanpercentile(np.abs(Ha), 99)
pa_v = np.nanpercentile(np.abs(Pa), 99)
fig, axes = plt.subplots(1,2, figsize=(8, 4))

im0 = axes[0].imshow(
    Ha,
    origin='lower',
    cmap='RdBu_r',
    vmin=-ha_v,
    vmax=ha_v
)
plt.colorbar(
            im0,
            ax=axes[0],
            fraction=0.046
        )
axes[0].axis('off')

im1 = axes[1].imshow(
    Pa,
    origin='lower',
    cmap='RdBu_r',
    vmin=-pa_v,
    vmax=pa_v
)
plt.colorbar(
            im1,
            ax=axes[1],
            fraction=0.046
        )
axes[1].axis('off')


axes[0].set_title('Ha')
axes[1].set_title('Pa')
plt.show()


In [ ]:
import Functions
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
import os
gal_names = ["ngc1433", 'ngc1512', 'ngc1672', "M51"]
galaxy_name = gal_names[3]


def continuum_subtract_f187(
    f150_file,
    f187_file,
    scale_factor,
    output_file=None
):
    """
    Continuum subtract a JWST F187N image using a scaled F150W image.

    Parameters
    ----------
    f150_file : str
        Path to the F150W FITS image.

    f187_file : str
        Path to the F187N FITS image.

    scale_factor : float
        Multiplicative scale factor applied to the F150W image.

    output_file : str, optional
        Output FITS filename.
        If None, a default filename is generated.

    Returns
    -------
    subtracted : ndarray
        Continuum-subtracted image array.
    """

    # ========================================================
    # LOAD DATA
    # ========================================================

    with fits.open(f150_file) as hdul150:
        f150_data = hdul150['SCI'].data.astype(float)
        f150_header = hdul150['SCI'].header

    with fits.open(f187_file) as hdul187:
        f187_data = hdul187['SCI'].data.astype(float)
        f187_header = hdul187['SCI'].header

    # ========================================================
    # CHECK SHAPES
    # ========================================================

    if f150_data.shape != f187_data.shape:
        raise ValueError(
            f'Image shapes do not match: '
            f'F150W {f150_data.shape} vs '
            f'F187N {f187_data.shape}'
        )

    # ========================================================
    # CONTINUUM SUBTRACTION
    # ========================================================

    continuum = scale_factor * f150_data

    subtracted = f187_data - continuum

    # ========================================================
    # OUTPUT FILENAME
    # ========================================================

    if output_file is None:

        base = os.path.splitext(os.path.basename(f187_file))[0]

        output_file = (
            f'{base}_contsub_scale{scale_factor:.5f}.fits'
        )

    # ========================================================
    # CREATE OUTPUT HEADER
    # ========================================================

    out_header = f187_header.copy()

    out_header['HISTORY'] = 'Continuum subtraction performed'
    out_header['CONTFILE'] = os.path.basename(f150_file)
    out_header['CONTSCL'] = scale_factor
    out_header['BUNIT'] = 'Continuum-subtracted flux'

    # ========================================================
    # WRITE FITS FILE
    # ========================================================

    hdu = fits.PrimaryHDU(
        data=subtracted,
        header=out_header
    )

    hdu.writeto(output_file, overwrite=True)

    print(f'Saved continuum-subtracted image:')
    print(f'    {output_file}')

    return subtracted

In [ ]:
%matplotlib widget

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from astropy.io import fits
from matplotlib.widgets import Slider

# ============================================================
# USER INPUTS
# ============================================================
galaxy_name = "M51"
if galaxy_name == "M51":
    hst_file = '/project/galaxies/tjuchau/data_files/HST/ngc5194/F555W_NGC5194_ACS_WFC_drc.fits'
    f150_file = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f150w_i2d_anchor.fits'
    f187_file = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_i2d_anchor.fits'
    f300_file = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f300m_i2d_anchor.fits'
    cont_sub_file = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_i2d_anchor_cont_subtracted.fits'
    out_file = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_cont_sub.fits'
elif galaxy_name == 'ngc1433':
    hst_file = '/project/galaxies/tjuchau/data_files/HST/ngc1433/hlsp_phangs-hst_hst_wfc3-uvis_ngc1433_f555w_v1_exp-drc-sci.fits'
    f150_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1433/ngc1433_nircam_lv3_f150w_i2d_anchor.fits'
    f187_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1433/ngc1433_nircam_lv3_f187n_i2d_anchor.fits'
    f300_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1433/ngc1433_nircam_lv3_f300m_i2d_anchor.fits'
    cont_sub_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1433/ngc1433_nircam_lv3_f187n_i2d_anchor_cont_subtracted.fits'
    out_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1433/ngc1433_nircam_lv3_f187n_cont_sub.fits'

elif galaxy_name == 'ngc1512':
    hst_file = '/project/galaxies/tjuchau/data_files/HST/ngc1512/hlsp_phangs-hst_hst_wfc3-uvis_ngc1512mosaic_f555w_v1_exp-drc-sci.fits'
    f150_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1512/ngc1512_nircam_lv3_f150w_i2d_anchor.fits'
    f187_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1512/ngc1512_nircam_lv3_f187n_i2d_anchor.fits'
    f300_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1512/ngc1512_nircam_lv3_f300m_i2d_anchor.fits'
    cont_sub_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1512/ngc1512_nircam_lv3_f187n_i2d_anchor_cont_subtracted.fits'
    out_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1512/ngc1512_nircam_lv3_f187n_cont_sub.fits'

elif galaxy_name == 'ngc1672':
    hst_file = '/project/galaxies/tjuchau/data_files/HST/ngc1672/hlsp_phangs-hst_hst_wfc3-uvis_ngc1672mosaic_f555w_v1_exp-drc-sci.fits'
    f150_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1672/ngc1672_nircam_lv3_f150w_i2d_anchor.fits'
    f187_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1672/ngc1672_nircam_lv3_f187n_i2d_anchor.fits'
    f300_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1672/ngc1672_nircam_lv3_f300m_i2d_anchor.fits'
    cont_sub_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1672/ngc1672_nircam_lv3_f187n_i2d_anchor_cont_subtracted.fits'
    out_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1672/ngc1672_nircam_lv3_f187n_cont_sub.fits'

#f150_file = '/project/galaxies/tjuchau/data_files/HST/ngc5194/F689M_HST_WFC3_UVIS_IVM_drc.fits'
#f187_file = '/project/galaxies/tjuchau/data_files/HST/ngc5194/F658N_HST_ACS_WFC_IVM_drc.fits'

try:
    f150 = fits.open(f150_file)['SCI'].data.astype(float)
    f187 = fits.open(f187_file)['SCI'].data.astype(float)
    f300 = fits.open(f300_file)['SCI'].data.astype(float)
except:
    f150 = fits.open(f150_file)[0].data.astype(float)
    f187 = fits.open(f187_file)[0].data.astype(float)
    f300 = fits.open(f300_file)[0].data.astype(float)
# Initial scale factor
initial_scale = 1.044

# Zoom region

# Set these manually after inspecting image size
x1, x2 = int(f150.shape[0]//2)-300, int(f150.shape[0]//2)+800
y1, y2 = int(f150.shape[1]//2)-300, int(f150.shape[1]//2)+800
mask_x = 5200
mask_y = 5200
mask_radius = 250

# ============================================================
# CREATE AGN MASK
# ============================================================

yy, xx = np.indices(f187.shape)

r = np.sqrt((xx - mask_x)**2 + (yy - mask_y)**2)

mask = r <= mask_radius

# Mask values with NaN
f187[mask] = np.nan
f150[mask] = np.nan

# ============================================================
# CUTOUT REGION
# ============================================================

f187_cut = f187[y1:y2, x1:x2]
f150_cut = f150[y1:y2, x1:x2]

# ============================================================
# INITIAL CONTINUUM MODEL
# ============================================================

continuum = initial_scale * f150_cut
subtracted = f187_cut - continuum

# ============================================================
# NORMALIZATION
# ============================================================

combined = np.concatenate([
    f187_cut[np.isfinite(f187_cut)].ravel(),
    continuum[np.isfinite(continuum)].ravel(),
    f150_cut[np.isfinite(f150_cut)].ravel()
])

vmin = np.percentile(combined, 1)
vmax = np.percentile(combined, 99.7)

sub_v = np.nanpercentile(np.abs(subtracted), 99)

# ============================================================
# FIGURE
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(8, 8))
axes = axes.ravel()
plt.subplots_adjust(bottom=0.15)

# ------------------------------------------------------------
# F187N
# ------------------------------------------------------------

im0 = axes[0].imshow(
    f187_cut,
    origin='lower',
    cmap='gray',
    vmin=vmin,
    vmax=vmax
)

axes[0].set_title('F187N')
axes[0].axis('off')

# ------------------------------------------------------------
# Continuum
# ------------------------------------------------------------

im1 = axes[1].imshow(
    continuum,
    origin='lower',
    cmap='gray',
    vmin=vmin,
    vmax=vmax
)

title1 = axes[1].set_title(f'Continuum = {initial_scale:.5f} × F150W')
axes[1].axis('off')

# ------------------------------------------------------------
# Subtracted
# ------------------------------------------------------------

im2 = axes[2].imshow(
    subtracted,
    origin='lower',
    cmap='RdBu_r',
    vmin=-sub_v,
    vmax=sub_v
)

title2 = axes[2].set_title('F187N - Continuum')
axes[2].axis('off')

# ------------------------------------------------------------
# f150
# ------------------------------------------------------------

im3 = axes[3].imshow(
    f150_cut,
    origin='lower',
    cmap='gray',
    vmin=-vmin,
    vmax=vmax
)

axes[3].set_title('F150W')
axes[3].axis('off')

# ============================================================
# COLORBARS
# ============================================================

plt.colorbar(im0, ax=axes[0], fraction=0.046)
plt.colorbar(im1, ax=axes[1], fraction=0.046)
plt.colorbar(im2, ax=axes[2], fraction=0.046)
plt.colorbar(im3, ax=axes[3], fraction=0.046)

# ============================================================
# SLIDER
# ============================================================

ax_slider = plt.axes([0.2, 0.05, 0.6, 0.03])

scale_slider = Slider(
    ax=ax_slider,
    label='Scale Factor',
    valmin=0.01,
    valmax=2,
    valinit=initial_scale,
    valstep=0.001
)

# ============================================================
# UPDATE FUNCTION
# ============================================================

def update(val):

    scale = scale_slider.val

    continuum_new = scale * f150_cut
    subtracted_new = f187_cut - continuum_new

    im1.set_data(continuum_new)
    im2.set_data(subtracted_new)

    sub_v_new = np.nanpercentile(np.abs(subtracted_new), 99)

    im2.set_clim(-sub_v_new, sub_v_new)

    title1.set_text(f'Continuum = {scale:.5f} × F150W')
    title2 = axes[2].set_title(f'F187N - Continuum <{np.nanmedian(subtracted_new)}>')

    fig.canvas.draw_idle()

scale_slider.on_changed(update)

plt.show()

In [ ]:
x1+915,y1+658